In [2]:
import h5py
import json
import numpy as np
from tqdm import tqdm
from mani_skill.utils.io_utils import load_json

In [3]:
def load_h5_data(data):
    out = dict()
    for k in data.keys():
        if isinstance(data[k], h5py.Dataset):
            out[k] = data[k][:]
        else:
            out[k] = load_h5_data(data[k])
    return out

def extract_mean_std_min_max(data_list):
    data = np.concatenate(data_list, axis=0)
    stats = {}
    means =np.mean(data, axis=0)
    stds = np.std(data, axis=0)
    mins = np.min(data, axis=0)
    maxs = np.max(data, axis=0)
    if data_list[0].ndim == 2 and data.shape[-1] > 1:
        mean = [float(mean) for mean in means]
        #mean = [float(mean[0]), float(mean[1]), float(mean[2])]
        std = [float(std) for std in stds]
        min = [float(min) for min in mins]
        max = [float(max) for max in maxs]
    else:
        mean =float(means)
        std = float(stds)
        min = float(mins)
        max = float(maxs)
    stats.update({"mean": mean})
    stats.update({"std": std})
    stats.update({"min": min})
    stats.update({"min": min})
    stats.update({"max": max})
    return stats


In [8]:
dataset_file = "/mnt/data_nrp/dataset/mani_skill_data/demos/PegInsertionSide-ExtendedIMG/motionplanning/trajectory.h5"
json_file_path = "/mnt/data_nrp/dataset/mani_skill_data/demos/PegInsertionSide-ExtendedIMG/motionplanning/stats.json"
json_path = dataset_file.replace(".h5", ".json")
json_data = load_json(json_path)
episodes = json_data["episodes"]
ds = h5py.File(dataset_file, "r")
key_stat_list = ["obs", "actions"]

In [9]:
actions_list = []
obs_dict_list = {
    "qpos": [],
    "qvel": [],
    "tcp_pose": [],
    "peg_pose": [],
    "box_hole_pose": [],
    #"time": []
}
for eps_id in tqdm(range(len(episodes))):
    eps = episodes[eps_id]
    trajectory = ds[f"traj_{eps['episode_id']}"]
    trajectory = load_h5_data(trajectory)
    actions_list.append(trajectory["actions"])
    obs= trajectory["obs"]
    for key_obs in obs.keys():
        # if key_obs == "time":
        #     obs_dict_list[key_obs].append(obs[key_obs])
        #     continue
        for key_i in obs[key_obs].keys():
            if key_i in obs_dict_list.keys():
                obs_dict_list[key_i].append(obs[key_obs][key_i])

100%|██████████| 1000/1000 [02:18<00:00,  7.24it/s]


In [11]:
json_stats = {}
json_stats["actions"] = extract_mean_std_min_max(actions_list)
json_stats["qpos"] = extract_mean_std_min_max(obs_dict_list["qpos"])
json_stats["qvel"] = extract_mean_std_min_max(obs_dict_list["qvel"])
json_stats["tcp_pose"] = extract_mean_std_min_max(obs_dict_list["tcp_pose"])
json_stats["peg_pose"] = extract_mean_std_min_max(obs_dict_list["peg_pose"])
json_stats["box_hole_pose"] = extract_mean_std_min_max(obs_dict_list["box_hole_pose"])

In [12]:
with open(json_file_path, "w") as f:
    json.dump(json_stats, f, indent=4)